# 01 - Dataset Exploration

This notebook verifies the prepared CamVid binary road-segmentation dataset and creates example figures for the report.

The binary task is:

- road = 255
- non-road = 0


## A. Imports and paths

Paths are defined relative to this notebook in the `notebooks/` folder.

In [ ]:
from pathlib import Path
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

PROJECT_ROOT = Path("..")
RAW_ROOT = PROJECT_ROOT / "data" / "raw" / "CamVid"
PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed"
FIGURES_ROOT = PROJECT_ROOT / "results" / "figures" / "dataset_examples"

FIGURES_ROOT.mkdir(parents=True, exist_ok=True)

SPLITS = ["train", "val", "test"]
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg"}
EXPECTED_MASK_VALUES = {0, 255}
RANDOM_SEED = 42


## B. Load dataset summary

The summary was generated during dataset preparation.

In [ ]:
summary_path = PROCESSED_ROOT / "dataset_summary.json"

with open(summary_path, "r", encoding="utf-8") as f:
    summary = json.load(f)

print(json.dumps(summary, indent=4))

summary_rows = []
for split_info in summary["splits"]:
    summary_rows.append(
        {
            "Split": split_info["split"],
            "Images": split_info["images_found"],
            "Converted Masks": split_info["converted_masks"],
            "Missing Labels": split_info["missing_labels"],
            "Road Pixel Ratio": f"{split_info['road_pixel_ratio']:.2%}",
        }
    )

pd.DataFrame(summary_rows)


## C. Show dataset split counts

This checks the number of input images, original colored labels, and prepared binary masks.

In [ ]:
def count_files(directory):
    return sorted(
        path
        for path in directory.iterdir()
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
    )


split_files = {}

for split in SPLITS:
    image_dir = RAW_ROOT / split
    label_dir = RAW_ROOT / f"{split}_labels"
    binary_mask_dir = PROCESSED_ROOT / "binary_masks" / split

    split_files[split] = {
        "images": count_files(image_dir),
        "labels": count_files(label_dir),
        "binary_masks": count_files(binary_mask_dir),
        "label_dir": label_dir,
        "binary_mask_dir": binary_mask_dir,
    }

    print(f"{split} images: {len(split_files[split]['images'])}")
    print(f"{split} original labels: {len(split_files[split]['labels'])}")
    print(f"{split} binary masks: {len(split_files[split]['binary_masks'])}")
    print()


## D. Display class_dict.csv

CamVid uses RGB colors to encode semantic classes in the original masks.

In [ ]:
class_dict_path = RAW_ROOT / "class_dict.csv"
class_dict = pd.read_csv(class_dict_path)

display(class_dict)

road_row = class_dict[class_dict["name"].str.strip().str.lower() == "road"].iloc[0]
road_rgb = (int(road_row["r"]), int(road_row["g"]), int(road_row["b"]))

print(f"Road = RGB{road_rgb}")


## G. Helper function for label matching

Different CamVid downloads may store labels with slightly different filename suffixes.

In [ ]:
def find_label_path(image_path, label_dir):
    candidates = [
        label_dir / image_path.name,
        label_dir / f"{image_path.stem}_L.png",
        label_dir / f"{image_path.stem}_label.png",
        label_dir / f"{image_path.stem}_gt.png",
        label_dir / f"{image_path.stem}_mask.png",
    ]

    for candidate in candidates:
        if candidate.exists():
            return candidate

    raise FileNotFoundError(f"No label found for {image_path.name}")


## E. Verify image and mask sizes

For one sample from each split, the input image, original colored mask, and binary mask should have the same spatial size.

In [ ]:
for split in SPLITS:
    image_path = split_files[split]["images"][0]
    label_path = find_label_path(image_path, split_files[split]["label_dir"])
    binary_mask_path = split_files[split]["binary_mask_dir"] / image_path.with_suffix(".png").name

    with Image.open(image_path) as image:
        image_size = image.size

    with Image.open(label_path) as label:
        label_size = label.size

    with Image.open(binary_mask_path) as binary_mask:
        binary_mask_size = binary_mask.size

    print(f"{split} sample: {image_path.name}")
    print(f"  input image size: {image_size}")
    print(f"  original colored mask size: {label_size}")
    print(f"  binary mask size: {binary_mask_size}")

    if image_size != label_size or image_size != binary_mask_size:
        raise ValueError(f"Size mismatch found for {split}: {image_path.name}")

    print()


## F. Verify binary mask values

Every prepared binary mask must contain only `0` and `255`.

In [ ]:
for split in SPLITS:
    observed_values = set()

    for mask_path in split_files[split]["binary_masks"]:
        mask_array = np.array(Image.open(mask_path))
        unique_values = set(np.unique(mask_array).astype(int).tolist())
        invalid_values = unique_values - EXPECTED_MASK_VALUES

        if invalid_values:
            raise ValueError(
                f"Invalid values in {mask_path}: {sorted(invalid_values)}. "
                "Expected only {0, 255}."
            )

        observed_values.update(unique_values)

    print(f"{split}: binary mask values = {sorted(observed_values)}")


## H. Show one dataset example

The binary mask is displayed with road pixels in white and non-road pixels in black.

In [ ]:
split = "train"
image_path = split_files[split]["images"][0]
label_path = find_label_path(image_path, split_files[split]["label_dir"])
binary_mask_path = split_files[split]["binary_mask_dir"] / image_path.with_suffix(".png").name

image = Image.open(image_path).convert("RGB")
label = Image.open(label_path).convert("RGB")
binary_mask = Image.open(binary_mask_path).convert("L")

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

axes[0].imshow(image)
axes[0].set_title("Input image")

axes[1].imshow(label)
axes[1].set_title("Original CamVid colored mask")

axes[2].imshow(binary_mask, cmap="gray", vmin=0, vmax=255)
axes[2].set_title("Binary road mask")

for ax in axes:
    ax.axis("off")

fig.suptitle(image_path.name)
fig.tight_layout()
plt.show()


## I. Show several random examples

These examples help catch obvious image-mask alignment or conversion issues before training models.

In [ ]:
random.seed(RANDOM_SEED)
sample_images = random.sample(split_files["train"]["images"], 4)

fig, axes = plt.subplots(len(sample_images), 3, figsize=(12, 3.5 * len(sample_images)))

for row_index, image_path in enumerate(sample_images):
    label_path = find_label_path(image_path, split_files["train"]["label_dir"])
    binary_mask_path = split_files["train"]["binary_mask_dir"] / image_path.with_suffix(".png").name

    image = Image.open(image_path).convert("RGB")
    label = Image.open(label_path).convert("RGB")
    binary_mask = Image.open(binary_mask_path).convert("L")

    panels = [
        (image, "Input image", None),
        (label, "Original CamVid mask", None),
        (binary_mask, "Binary road mask", "gray"),
    ]

    for col_index, (panel_image, title, cmap) in enumerate(panels):
        ax = axes[row_index, col_index]
        if cmap == "gray":
            ax.imshow(panel_image, cmap=cmap, vmin=0, vmax=255)
        else:
            ax.imshow(panel_image)
        ax.set_title(title)
        ax.axis("off")

fig.tight_layout()
plt.show()


## J. Save 10 dataset example figures

Saved figures can be used directly in the final report and presentation.

In [ ]:
random.seed(RANDOM_SEED)
example_images = random.sample(split_files["train"]["images"], 10)

for index, image_path in enumerate(example_images, start=1):
    label_path = find_label_path(image_path, split_files["train"]["label_dir"])
    binary_mask_path = split_files["train"]["binary_mask_dir"] / image_path.with_suffix(".png").name

    image = Image.open(image_path).convert("RGB")
    label = Image.open(label_path).convert("RGB")
    binary_mask = Image.open(binary_mask_path).convert("L")

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    panels = [
        (image, "Input image", None),
        (label, "Original CamVid mask", None),
        (binary_mask, "Binary road mask", "gray"),
    ]

    for ax, (panel_image, title, cmap) in zip(axes, panels):
        if cmap == "gray":
            ax.imshow(panel_image, cmap=cmap, vmin=0, vmax=255)
        else:
            ax.imshow(panel_image)
        ax.set_title(title)
        ax.axis("off")

    fig.suptitle(image_path.name)
    fig.tight_layout()

    output_path = FIGURES_ROOT / f"dataset_example_{index:02d}.png"
    fig.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.close(fig)

print(f"Saved 10 dataset example figures to: {FIGURES_ROOT}")
